# Bài học 17 - Tạo tác nhân AI cục bộ với Foundry Local và Qwen

Trong sổ ghi chép này, bạn xây dựng một **trợ lý kỹ thuật cục bộ** chạy hoàn toàn trên máy trạm của bạn. Không sử dụng suy luận trên đám mây ở bất kỳ điểm nào. Trợ lý có thể:

1. **Gọi công cụ** qua gọi hàm Qwen thông qua Foundry Local.
2. **Liệt kê và đọc các tập tin** bên trong thư mục dự án được cô lập.
3. **Phân tích mã** với các chỉ số đơn giản.
4. **Tìm kiếm tài liệu** với RAG cục bộ (Chroma).
5. **Sử dụng máy chủ MCP cục bộ** (bỏ qua nhẹ nhàng nếu không có cấu hình).

Mã tác nhân gần như giống hệt các bài học trên đám mây — chỉ khác điểm cuối của client chuyển từ đám mây sang `localhost`.


## Thiết lập

Trước khi chạy sổ tay này:

1. **Cài đặt Microsoft Foundry Local** (xem [tài liệu](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) dành cho hệ điều hành của bạn).
2. **Tải xuống và khởi động một mô hình Qwen:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. Cài đặt các gói Python bên dưới.

Mọi thứ hoạt động cục bộ. Một máy có khoảng 8 GB RAM là tối thiểu hợp lý.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Kết nối với Foundry Local

`FoundryLocalManager` tải mô hình nếu cần, khởi động dịch vụ cục bộ, và cung cấp cho chúng ta một **điểm cuối tương thích với OpenAI**. Sau đó, chúng ta trỏ SDK OpenAI chuẩn đến đó. Khóa API là một chỗ giữ chỗ cục bộ — không liên quan đến chứng thực đám mây.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. Công cụ cục bộ (Hoạt động tệp trong vùng cách ly)

Chúng tôi tạo một dự án mẫu nhỏ trên đĩa, sau đó định nghĩa các công cụ giới hạn trong thư mục gốc của dự án đó. Việc kiểm tra vùng cách ly vẫn quan trọng ngay cả ở cấp độ cục bộ: một công cụ đọc các đường dẫn tùy ý sẽ chạy với quyền của người dùng của bạn và có thể truy cập mọi thứ bạn có thể.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. RAG cục bộ với Chroma

Chúng tôi nhúng một bộ nhỏ các đoạn tài liệu vào bộ sưu tập Chroma cục bộ. Chroma chạy trong quy trình và lưu vectors trên đĩa — không cần máy chủ, không cần đám mây. Công cụ `search_docs` truy xuất các đoạn có liên quan nhất cho một truy vấn.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. Vòng lặp gọi công cụ

Bây giờ chúng ta đăng ký các công cụ với mô hình bằng cách sử dụng schema công cụ OpenAI và chạy vòng lặp gọi công cụ tiêu chuẩn — mô hình yêu cầu một công cụ, chúng ta thực thi nó tại chỗ, cung cấp kết quả quay lại, và lặp lại cho đến khi mô hình đưa ra câu trả lời cuối cùng. Khả năng gọi hàm đáng tin cậy của Qwen là điều khiến điều này hoạt động trên thiết bị.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. MCP cục bộ (Tùy chọn)

MCP là một phương tiện truyền tải, không phải dịch vụ đám mây — một máy chủ MCP có thể chạy như một tiến trình cục bộ qua `stdio`. Ô dưới đây cho thấy cách bạn kết nối với một máy chủ MCP cục bộ nếu bạn đã cấu hình một máy chủ như vậy (ví dụ một máy chủ hệ thống tập tin). Nó sẽ bỏ qua một cách nhẹ nhàng khi `LOCAL_MCP_COMMAND` không được thiết lập, vì vậy sổ tay vẫn chạy trọn vẹn mà không cần nó.

Lưu ý về bảo mật: một máy chủ MCP cục bộ chạy với quyền người dùng của bạn. Giới hạn nó trong thư mục dự án và xác thực đầu ra của nó trước khi thực hiện hành động dựa trên chúng.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## Tóm tắt

Bạn đã xây dựng một trợ lý kỹ thuật vận hành hoàn toàn trên máy của bạn:

- **Foundry Local** phục vụ một mô hình **Qwen** phía sau một endpoint tương thích với OpenAI — vì vậy mã tác nhân phù hợp với các bài học trên đám mây.
- **Công cụ Sandboxed** cung cấp cho tác nhân quyền truy cập tệp và phân tích mã mà không rời khỏi thư mục dự án.
- **Chroma** cung cấp **RAG cục bộ** trên tài liệu.
- **Local MCP** cho thấy cách tái sử dụng hệ sinh thái MCP ngoại tuyến.

Không sử dụng suy luận đám mây tại bất kỳ thời điểm nào.

### Thách thức

Mở rộng tác nhân cục bộ để:

1. **Làm việc với nhiều máy chủ MCP** — kết nối một máy chủ hệ thống tệp và một máy chủ git và để tác nhân chọn giữa chúng.
2. **Sử dụng bộ nhớ cục bộ** — lưu trữ lịch sử hội thoại ngắn lên đĩa để trợ lý nhớ được các lượt trước đó qua các lần khởi động lại sổ tay.
3. **Hỗ trợ phối hợp đa tác nhân cục bộ** — thêm một tác nhân cục bộ thứ hai (ví dụ như người đánh giá) và để hai tác nhân hợp tác trong một nhiệm vụ.

Trong bài học tiếp theo, bạn sẽ học cách bảo mật các tác nhân AI đã triển khai.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
